# Project 1 — Robust Bank Accounts

**Python 3.11+** (no pin to the original notebook's Python 3.6). Only the standard library is required. Run cells from top to bottom.

Design decisions: `Decimal` for currency; two decimal places with `ROUND_HALF_EVEN`; strictly positive deposit/withdrawal amounts; declined withdrawals return an `X` confirmation and retain the balance; invalid input raises `ValueError`. Account identifiers are immutable, digit-only **strings** so leading zeros survive. The global transaction sequence starts at 1 per process. All transaction timestamps are UTC and second-resolution. Fixed-offset time zones do **not** implement daylight-saving transitions.

**Correction to the prompt:** 0.5% of 1,000 is 5, so the result is **1,005**. The stated **1,050** result corresponds to a **5%** rate.

This is a teaching implementation, not a production banking system: production use requires durable ACID storage, account authorization, idempotency, persistent globally unique IDs, and a real audit system.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone, tzinfo
from decimal import Decimal, InvalidOperation, ROUND_HALF_EVEN
from itertools import count
from threading import Lock, RLock
from typing import ClassVar
import re

CENT = Decimal("0.01")
CONFIRMATION_PATTERN = re.compile(
    r"(?P<code>[DWIX])-(?P<account>[0-9]+)-(?P<stamp>[0-9]{14})-(?P<id>[1-9][0-9]*)"
)


def as_decimal(value: Decimal | str | int | float, *, label: str) -> Decimal:
    """Accept conventional numeric inputs; convert floats via str, never binary value."""
    if isinstance(value, bool) or not isinstance(value, (Decimal, str, int, float)):
        raise TypeError(f"{label} must be a Decimal, string, int, or float")
    try:
        result = Decimal(str(value))
    except (InvalidOperation, ValueError) as exc:
        raise ValueError(f"{label} must be a valid number") from exc
    if not result.is_finite():
        raise ValueError(f"{label} must be finite")
    return result


def money(value: Decimal | str | int | float, *, label: str) -> Decimal:
    """Reject amounts with sub-cent precision instead of silently rounding them."""
    result = as_decimal(value, label=label)
    try:
        rounded = result.quantize(CENT, rounding=ROUND_HALF_EVEN)
    except InvalidOperation as exc:
        raise ValueError(f"{label} cannot be represented as currency") from exc
    if result != rounded:
        raise ValueError(f"{label} cannot contain fractions of a cent")
    return rounded

In [2]:
@dataclass(frozen=True, slots=True)
class TimeZone:
    """Named fixed UTC offset. For DST-sensitive locations use zoneinfo instead."""

    name: str
    offset_hours: int = 0
    offset_minutes: int = 0

    def __post_init__(self) -> None:
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("Time zone name must be nonempty")
        if type(self.offset_hours) is not int or type(self.offset_minutes) is not int:
            raise TypeError("Time zone offsets must be integers")
        if abs(self.offset_minutes) >= 60:
            raise ValueError("Minute offset magnitude must be below 60")
        if self.offset_hours and self.offset_minutes and (
            (self.offset_hours > 0) != (self.offset_minutes > 0)
        ):
            raise ValueError("Hour and minute offsets must have matching signs")
        total_minutes = self.offset_hours * 60 + self.offset_minutes
        if abs(total_minutes) > 14 * 60:
            raise ValueError("UTC offset must fall between -14:00 and +14:00")
        object.__setattr__(self, "name", self.name.strip())

    @property
    def tzinfo(self) -> timezone:
        return timezone(
            timedelta(hours=self.offset_hours, minutes=self.offset_minutes),
            name=self.name,
        )


UTC = TimeZone("UTC")
MST = TimeZone("MST", -7)


@dataclass(frozen=True, slots=True)
class Confirmation:
    account_number: str
    transaction_code: str
    transaction_id: int
    timestamp_utc: datetime
    timestamp_local: datetime

    @property
    def time_utc(self) -> str:
        return self.timestamp_utc.strftime("%Y-%m-%dT%H:%M:%S")

    @property
    def time(self) -> str:
        name = self.timestamp_local.tzname() or self.timestamp_local.strftime("%z")
        return f"{self.timestamp_local:%Y-%m-%d %H:%M:%S} ({name})"


@dataclass(frozen=True, slots=True)
class Transaction:
    transaction_id: int
    code: str
    amount: Decimal
    balance_after: Decimal
    timestamp_utc: datetime
    confirmation: str
    reason: str | None = None

In [3]:
class Account:
    """In-memory account; public mutation only through validated transactions."""

    _monthly_interest_rate: ClassVar[Decimal] = Decimal("0.005")
    _rate_lock: ClassVar[Lock] = Lock()
    _id_lock: ClassVar[Lock] = Lock()
    _transaction_ids: ClassVar[count] = count(1)

    def __init__(
        self,
        account_number: str | int,
        first_name: str,
        last_name: str,
        preferred_timezone: TimeZone = UTC,
        starting_balance: Decimal | str | int | float = "0.00",
    ) -> None:
        if isinstance(account_number, bool) or not isinstance(account_number, (str, int)):
            raise TypeError("Account number must be a digit-only string or integer")
        identifier = str(account_number)
        if not identifier or not identifier.isascii() or not identifier.isdecimal():
            raise ValueError("Account number must contain only ASCII digits")
        if not isinstance(preferred_timezone, TimeZone):
            raise TypeError("preferred_timezone must be a TimeZone")
        initial = money(starting_balance, label="Starting balance")
        if initial < 0:
            raise ValueError("Starting balance cannot be negative")

        self._account_number = identifier
        self._first_name = self._validate_name(first_name, "First name")
        self._last_name = self._validate_name(last_name, "Last name")
        self._preferred_timezone = preferred_timezone
        self._balance = initial
        self._transactions: list[Transaction] = []
        self._lock = RLock()

    @staticmethod
    def _validate_name(value: str, label: str) -> str:
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{label} must be a nonempty string")
        return value.strip()

    @staticmethod
    def _utcnow() -> datetime:
        return datetime.now(timezone.utc)

    @classmethod
    def _next_id(cls) -> int:
        # Refer to Account explicitly: all subclasses must share one sequence.
        with Account._id_lock:
            return next(Account._transaction_ids)

    @classmethod
    def get_monthly_interest_rate(cls) -> Decimal:
        with Account._rate_lock:
            return Account._monthly_interest_rate

    @classmethod
    def set_monthly_interest_rate(cls, rate: Decimal | str | int | float) -> None:
        """Set the shared *fractional* rate: 0.005 means 0.5%."""
        rate_value = as_decimal(rate, label="Monthly interest rate")
        if rate_value < 0:
            raise ValueError("Monthly interest rate cannot be negative")
        with Account._rate_lock:
            Account._monthly_interest_rate = rate_value

    @property
    def interest_rate(self) -> Decimal:
        return self.get_monthly_interest_rate()

    @property
    def account_number(self) -> str:
        return self._account_number

    @property
    def first_name(self) -> str:
        return self._first_name

    @first_name.setter
    def first_name(self, value: str) -> None:
        self._first_name = self._validate_name(value, "First name")

    @property
    def last_name(self) -> str:
        return self._last_name

    @last_name.setter
    def last_name(self, value: str) -> None:
        self._last_name = self._validate_name(value, "Last name")

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    @property
    def preferred_timezone(self) -> TimeZone:
        return self._preferred_timezone

    @preferred_timezone.setter
    def preferred_timezone(self, value: TimeZone) -> None:
        if not isinstance(value, TimeZone):
            raise TypeError("preferred_timezone must be a TimeZone")
        self._preferred_timezone = value

    @property
    def balance(self) -> Decimal:
        with self._lock:
            return self._balance

    @property
    def transactions(self) -> tuple[Transaction, ...]:
        """Immutable snapshot of the in-memory audit history."""
        with self._lock:
            return tuple(self._transactions)

    def _record(self, code: str, amount: Decimal, *, reason: str | None = None) -> str:
        """Called only while the account lock is held."""
        timestamp = self._utcnow().astimezone(timezone.utc).replace(microsecond=0)
        transaction_id = self._next_id()
        confirmation = (
            f"{code}-{self.account_number}-{timestamp:%Y%m%d%H%M%S}-{transaction_id}"
        )
        self._transactions.append(
            Transaction(transaction_id, code, amount, self._balance, timestamp, confirmation, reason)
        )
        return confirmation

    def deposit(self, amount: Decimal | str | int | float) -> str:
        value = money(amount, label="Deposit")
        if value <= 0:
            raise ValueError("Deposit must be strictly positive")
        with self._lock:
            self._balance += value
            return self._record("D", value)

    def withdraw(self, amount: Decimal | str | int | float) -> str:
        value = money(amount, label="Withdrawal")
        if value <= 0:
            raise ValueError("Withdrawal must be strictly positive")
        with self._lock:
            if value > self._balance:
                return self._record("X", value, reason="Insufficient funds")
            self._balance -= value
            return self._record("W", value)

    def pay_interest(self) -> str:
        """Credit shared monthly interest; record I even for a zero balance."""
        with self._lock:
            rate = self.get_monthly_interest_rate()
            try:
                interest = (self._balance * rate).quantize(CENT, rounding=ROUND_HALF_EVEN)
            except InvalidOperation as exc:
                raise ValueError("Interest cannot be represented as currency") from exc
            self._balance += interest
            return self._record("I", interest)

    @staticmethod
    def parse_confirmation(
        confirmation: str, desired_timezone: TimeZone | tzinfo = UTC
    ) -> Confirmation:
        """No Account instance is required: confirmation encodes every needed field."""
        if not isinstance(confirmation, str):
            raise TypeError("Confirmation must be a string")
        match = CONFIRMATION_PATTERN.fullmatch(confirmation)
        if match is None:
            raise ValueError("Malformed confirmation number")
        zone = (
            desired_timezone.tzinfo
            if isinstance(desired_timezone, TimeZone)
            else desired_timezone
        )
        if not isinstance(zone, tzinfo):
            raise TypeError("desired_timezone must be TimeZone or datetime.tzinfo")
        try:
            stamp = datetime.strptime(match["stamp"], "%Y%m%d%H%M%S").replace(
                tzinfo=timezone.utc
            )
        except ValueError as exc:
            raise ValueError("Confirmation contains an invalid UTC timestamp") from exc
        return Confirmation(
            account_number=match["account"],
            transaction_code=match["code"],
            transaction_id=int(match["id"]),
            timestamp_utc=stamp,
            timestamp_local=stamp.astimezone(zone),
        )

## Usage and requirements checks

The parser is a `staticmethod` because its inputs are sufficient: the confirmation embeds account number, code, UTC timestamp and transaction ID; the caller supplies the desired timezone. The parsed record is a **structural interpretation**, not proof that a transaction occurred. To verify authenticity, look up the confirmation in a trusted persistent ledger (outside this exercise).

In [4]:
account = Account("140568", "Ada", "Lovelace", MST, "100.00")
deposit_code = account.deposit("50.00")
parsed = Account.parse_confirmation(deposit_code, MST)
print("Holder:", account.full_name)
print("Balance:", account.balance)
print("Confirmation:", deposit_code)
print("Parsed:", parsed)
print("Local time:", parsed.time)
print("UTC time:", parsed.time_utc)
print("Declined:", account.withdraw("999.00"))
print("Balance after decline:", account.balance)
print("Ledger entries:", len(account.transactions))

# The example's expected interest balance requires 5%, not 0.5%.
interest_account = Account("000140569", "Grace", "Hopper", starting_balance="1000.00")
previous_rate = Account.get_monthly_interest_rate()
try:
    Account.set_monthly_interest_rate("0.005")
    print("0.5% interest confirmation:", interest_account.pay_interest())
    print("Balance after 0.5%:", interest_account.balance)  # 1005.00
    Account.set_monthly_interest_rate("0.05")
    example_account = Account("000140570", "Katherine", "Johnson", starting_balance="1000.00")
    print("5% interest confirmation:", example_account.pay_interest())
    print("Balance after 5%:", example_account.balance)  # 1050.00
finally:
    Account.set_monthly_interest_rate(previous_rate)

Holder: Ada Lovelace
Balance: 150.00
Confirmation: D-140568-20260920123449-1
Parsed: Confirmation(account_number='140568', transaction_code='D', transaction_id=1, timestamp_utc=datetime.datetime(2026, 9, 20, 12, 34, 49, tzinfo=datetime.timezone.utc), timestamp_local=datetime.datetime(2026, 9, 20, 5, 34, 49, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200), 'MST')))
Local time: 2026-09-20 05:34:49 (MST)
UTC time: 2026-09-20T12:34:49
Declined: X-140568-20260920123449-2
Balance after decline: 150.00
Ledger entries: 2
0.5% interest confirmation: I-000140569-20260920123449-3
Balance after 0.5%: 1005.00
5% interest confirmation: I-000140570-20260920123449-4
Balance after 5%: 1050.00


## Automated tests

The tests avoid asserting absolute transaction IDs because IDs are shared across all accounts and previous cells may already have generated transactions. A concurrency test checks atomic balance updates and global ID uniqueness.

In [5]:
import unittest
from concurrent.futures import ThreadPoolExecutor
from unittest.mock import patch


class AccountTests(unittest.TestCase):
    def setUp(self) -> None:
        self.previous_rate = Account.get_monthly_interest_rate()
        Account.set_monthly_interest_rate("0.005")
        self.account = Account("00140568", "Ada", "Lovelace", MST, "100.00")

    def tearDown(self) -> None:
        Account.set_monthly_interest_rate(self.previous_rate)

    def test_names_read_only_fields_and_leading_zeros(self) -> None:
        self.assertEqual(self.account.account_number, "00140568")
        self.account.first_name = " Grace "
        self.assertEqual(self.account.full_name, "Grace Lovelace")
        with self.assertRaises(AttributeError):
            self.account.balance = Decimal("999")
        with self.assertRaises(AttributeError):
            self.account.account_number = "12"
        with self.assertRaises(ValueError):
            self.account.last_name = "  "

    def test_deposit_and_withdrawal(self) -> None:
        deposit = self.account.deposit("50")
        withdrawal = self.account.withdraw("25")
        self.assertTrue(deposit.startswith("D-00140568-"))
        self.assertTrue(withdrawal.startswith("W-00140568-"))
        self.assertEqual(self.account.balance, Decimal("125.00"))

    def test_decline_preserves_balance_and_records_reason(self) -> None:
        code = self.account.withdraw("100.01")
        self.assertTrue(code.startswith("X-"))
        self.assertEqual(self.account.balance, Decimal("100.00"))
        self.assertEqual(self.account.transactions[-1].reason, "Insufficient funds")

    def test_invalid_money_and_negative_opening_balance(self) -> None:
        for bad in ("0", "-1", "0.001", "NaN", float("inf")):
            with self.subTest(bad=bad), self.assertRaises(ValueError):
                self.account.deposit(bad)
        with self.assertRaises(ValueError):
            Account("123", "A", "B", starting_balance="-1")
        with self.assertRaises(ValueError):
            self.account.withdraw("0")
        with self.assertRaises(TypeError):
            self.account.deposit(True)

    def test_interest_is_shared_and_correct(self) -> None:
        other = Account("77", "Grace", "Hopper", starting_balance="1000.00")
        Account.set_monthly_interest_rate("0.05")
        self.assertEqual(self.account.interest_rate, other.interest_rate)
        code = other.pay_interest()
        self.assertTrue(code.startswith("I-"))
        self.assertEqual(other.balance, Decimal("1050.00"))
        Account.set_monthly_interest_rate("0.005")
        another = Account("78", "Katherine", "Johnson", starting_balance="1000")
        another.pay_interest()
        self.assertEqual(another.balance, Decimal("1005.00"))
        with self.assertRaises(ValueError):
            Account.set_monthly_interest_rate("-0.1")

    def test_half_even_interest_rounding(self) -> None:
        Account.set_monthly_interest_rate("0.025")
        small = Account("79", "A", "B", starting_balance="0.20")
        small.pay_interest()  # 0.005 is exactly half a cent -> 0.00 (half even)
        self.assertEqual(small.balance, Decimal("0.20"))

    def test_parse_known_timestamp_and_fixed_timezone(self) -> None:
        fixed = datetime(2019, 3, 15, 14, 59, tzinfo=timezone.utc)
        with patch.object(Account, "_utcnow", return_value=fixed):
            code = self.account.deposit("50")
        parsed = Account.parse_confirmation(code, MST)
        self.assertEqual(parsed.account_number, "00140568")
        self.assertEqual(parsed.transaction_code, "D")
        self.assertEqual(parsed.time_utc, "2019-03-15T14:59:00")
        self.assertEqual(parsed.time, "2019-03-15 07:59:00 (MST)")
        self.assertEqual(parsed.timestamp_utc.tzinfo, timezone.utc)

    def test_parser_rejects_invalid_inputs(self) -> None:
        for invalid in ("bad", "D-1-20190230010101-1", "Z-1-20200101000000-1"):
            with self.subTest(invalid=invalid), self.assertRaises(ValueError):
                Account.parse_confirmation(invalid)
        with self.assertRaises(TypeError):
            Account.parse_confirmation("D-1-20200101000000-1", "MST")

    def test_timezone_validation_and_immutable_ledger(self) -> None:
        with self.assertRaises(ValueError):
            TimeZone("bad", 15)
        with self.assertRaises(ValueError):
            TimeZone("bad", -3, 30)
        self.account.deposit("1")
        snapshot = self.account.transactions
        self.assertIsInstance(snapshot, tuple)
        with self.assertRaises(AttributeError):
            snapshot[0].amount = Decimal("900")

    def test_global_ids_and_thread_safe_balance(self) -> None:
        a = Account("100", "A", "B")
        b = Account("200", "C", "D")
        with ThreadPoolExecutor(max_workers=8) as pool:
            confirmations = list(pool.map(lambda i: (a if i % 2 else b).deposit("1"), range(200)))
        ids = [Account.parse_confirmation(code).transaction_id for code in confirmations]
        self.assertEqual(len(set(ids)), 200)
        self.assertEqual(a.balance, Decimal("100.00"))
        self.assertEqual(b.balance, Decimal("100.00"))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(AccountTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "A unit test failed"

test_decline_preserves_balance_and_records_reason (__main__.AccountTests.test_decline_preserves_balance_and_records_reason) ... ok
test_deposit_and_withdrawal (__main__.AccountTests.test_deposit_and_withdrawal) ... ok
test_global_ids_and_thread_safe_balance (__main__.AccountTests.test_global_ids_and_thread_safe_balance) ... ok
test_half_even_interest_rounding (__main__.AccountTests.test_half_even_interest_rounding) ... ok
test_interest_is_shared_and_correct (__main__.AccountTests.test_interest_is_shared_and_correct) ... ok
test_invalid_money_and_negative_opening_balance (__main__.AccountTests.test_invalid_money_and_negative_opening_balance) ... ok
test_names_read_only_fields_and_leading_zeros (__main__.AccountTests.test_names_read_only_fields_and_leading_zeros) ... ok
test_parse_known_timestamp_and_fixed_timezone (__main__.AccountTests.test_parse_known_timestamp_and_fixed_timezone) ... ok
test_parser_rejects_invalid_inputs (__main__.AccountTests.test_parser_rejects_invalid_inputs) ... 

## Notes for a production implementation

- The underscore-prefixed attributes implement **conventional** encapsulation, not a Python security boundary. Consumers should use the public API.
- An in-memory counter resets between runs and is not unique across processes. In production, allocate IDs transactionally in a database and persist the balance and immutable ledger atomically.
- Checking confirmation syntax does not authenticate it; a signed receipt or trusted ledger lookup would be necessary.
- Fixed UTC offsets such as `MST = UTC−07:00` do not handle daylight-saving time. For real locations, prefer `zoneinfo.ZoneInfo("America/Denver")` with the IANA timezone database.
- A live multi-currency banking application would also need currency metadata, authorization, transfer atomicity, duplicate-request protection, appropriate compliance controls, and policies for rate-effective dates and accrual periods.